# Headline Manipulation Classifier
Fine-tunes DistilBERT to detect manipulative framing in headlines and social posts.

**Runtime:** GPU recommended. In Colab: Runtime → Change runtime type → T4 GPU (free tier)

**Output:** An ONNX model ready to run in a Chrome extension via Transformers.js

**Label:** `1 = manipulative`, `0 = not manipulative`

## 1. Install dependencies

In [1]:
!pip install -q --upgrade datasets
!pip install -q transformers torch scikit-learn "optimum[exporters]" onnxruntime seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 14.4 MB/s eta 0:00:00


## 2. Imports

In [2]:
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## 3. Load Webis Clickbait Corpus 2017

38k social media posts labeled by human annotators with a clickbait score (0–1).  
We threshold at 0.5 → binary label.

In [3]:
try:
    webis_raw = load_dataset("webis/clickbait_17")
    print("Loaded from HuggingFace:")
    print(webis_raw)
    print("\nSample:", webis_raw['train'][0])
    WEBIS_AVAILABLE = True
except Exception as e:
    print(f"Webis load failed: {e}")
    print("\nProceeding without Webis — christinacdl dataset will be used as the primary source.")
    webis_raw = None
    WEBIS_AVAILABLE = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Webis load failed: Dataset 'webis/clickbait_17' doesn't exist on the Hub or cannot be accessed.

Proceeding without Webis — christinacdl dataset will be used as the primary source.


In [4]:
# --- ONLY run this cell if the HuggingFace load above failed ---
# Uncomment and run after uploading the .jsonl files manually

# import json
#
# def load_jsonl(path):
#     with open(path) as f:
#         return [json.loads(line) for line in f]
#
# train_raw = load_jsonl('clickbait17-train.jsonl')
# test_raw  = load_jsonl('clickbait17-test.jsonl')
#
# def parse_webis(records):
#     rows = []
#     for r in records:
#         text = ' '.join(r.get('postText', [])) or r.get('targetTitle', '')
#         score = r.get('truthMean', r.get('clickbaitScore', 0))
#         rows.append({'text': text.strip(), 'label': int(float(score) >= 0.5)})
#     return pd.DataFrame(rows)
#
# df_train = parse_webis(train_raw)
# df_test  = parse_webis(test_raw)
# print(df_train['label'].value_counts())

In [5]:
def webis_to_df(split):
    rows = []
    for item in split:
        post  = item.get('postText', [])
        text  = ' '.join(post) if isinstance(post, list) else str(post)
        if not text.strip():
            text = item.get('targetTitle', '')
        score = item.get('truthMean', item.get('clickbaitScore', 0))
        rows.append({'text': text.strip(), 'label': int(float(score) >= 0.5)})
    return pd.DataFrame(rows)

if WEBIS_AVAILABLE:
    df_train = webis_to_df(webis_raw['train'])
    df_val   = webis_to_df(webis_raw['validation']) if 'validation' in webis_raw else None
    df_test  = webis_to_df(webis_raw['test'])
    print(f"Webis — Train: {len(df_train)}  |  Test: {len(df_test)}")
    print(df_train['label'].value_counts())
else:
    df_train = pd.DataFrame(columns=['text', 'label'])
    df_val   = None
    df_test  = pd.DataFrame(columns=['text', 'label'])
    print("Webis skipped — df_train/df_test are empty placeholders.")

Webis skipped — df_train/df_test are empty placeholders.


## 3b. Load clickbait dataset

Tries several HuggingFace datasets in order until one loads successfully.

In [6]:
CANDIDATES = [
    ("christinacdl/clickbait_detection_dataset", "text",  "label"),
    ("marksverdhei/clickbait_title_classification", "title", "label"),
    ("evoreign/clickbait_headline",               "text",  "label"),
]

df_christinacdl_train = pd.DataFrame(columns=['text', 'label'])

for dataset_id, text_col, label_col in CANDIDATES:
    try:
        raw = load_dataset(dataset_id)
        print(f"Loaded {dataset_id}: {raw}")

        def to_df(split):
            rows = []
            for item in split:
                text  = str(item.get(text_col, '')).strip()
                label = int(item.get(label_col, 0))
                if text:
                    rows.append({'text': text, 'label': label})
            return pd.DataFrame(rows)

        first_key = list(raw.keys())[0]
        all_items = raw[first_key]

        # Handle datasets that encode split as a column rather than separate keys
        if 'split' in all_items.column_names:
            df_tmp = to_df(all_items)
            df_tmp['_split'] = [item['split'] for item in all_items]
            df_christinacdl_train = df_tmp[df_tmp['_split'] == 'train'][['text', 'label']].reset_index(drop=True)
            if len(df_christinacdl_train) == 0:
                df_christinacdl_train = df_tmp[['text', 'label']]
        elif 'train' in raw:
            df_christinacdl_train = to_df(raw['train'])
        else:
            df_christinacdl_train = to_df(all_items)

        if len(df_christinacdl_train) > 0:
            print(f"\nUsing {dataset_id} — {len(df_christinacdl_train)} train examples")
            print(df_christinacdl_train['label'].value_counts())
            break
        else:
            print(f"{dataset_id} loaded but was empty, trying next...")

    except Exception as e:
        print(f"{dataset_id} failed: {e}")

if len(df_christinacdl_train) == 0:
    print("\nAll candidates failed. Will train on Webis only (or fail if Webis also unavailable).")

df_christinacdl_train.head()

README.md:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

train.json:   0%|          | 0.00/2.99M [00:00<?, ?B/s]

val.json:   0%|          | 0.00/375k [00:00<?, ?B/s]

test.json:   0%|          | 0.00/373k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30296 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3787 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3787 [00:00<?, ? examples/s]

Loaded christinacdl/clickbait_detection_dataset: DatasetDict({
    train: Dataset({
        features: ['text_label', 'text', 'label'],
        num_rows: 30296
    })
    validation: Dataset({
        features: ['text_label', 'text', 'label'],
        num_rows: 3787
    })
    test: Dataset({
        features: ['text_label', 'text', 'label'],
        num_rows: 3787
    })
})

Using christinacdl/clickbait_detection_dataset — 30296 train examples
label
1    16016
0    14280
Name: count, dtype: int64


,text,label
0,14 Times People Turned Into Emojis At The Munk...,1
1,BBC to cut Electric Proms for financial reasons,0
2,28 Adorable Animal Things You Need In Your Life,1
3,Sotomayor on the Issues,0
4,9 Ways To Save Your Sanity On The Wednesday Be...,1


## 3c. Load LIAR dataset

10k political statements labeled for truthfulness (6 classes).  
Maps to binary: false/barely-true/pants-fire → manipulative, mostly-true/true → legitimate.  
Half-true is skipped as too ambiguous.  
Adds misinformation and deceptive framing coverage.

In [ ]:
# LIAR label mapping:
# 0=false, 1=half-true, 2=mostly-true, 3=true, 4=barely-true, 5=pants-fire
# manipulative (1): false(0), barely-true(4), pants-fire(5)
# legitimate  (0): mostly-true(2), true(3)
# skipped         : half-true(1) — too ambiguous
LIAR_MANIPULATIVE = {0, 4, 5}
LIAR_LEGITIMATE   = {2, 3}

df_liar_train = pd.DataFrame(columns=['text', 'label'])

try:
    for liar_id in ['ucsbnlp/liar', 'liar']:
        try:
            liar_raw = load_dataset(liar_id)
            print(f'Loaded {liar_id}:')
            print(liar_raw)
            break
        except Exception as e:
            print(f'{liar_id} failed: {e}')
            liar_raw = None

    if liar_raw is not None:
        def liar_to_df(split):
            rows = []
            for item in split:
                lbl = int(item['label'])
                if lbl in LIAR_MANIPULATIVE:
                    rows.append({'text': str(item['statement']).strip(), 'label': 1})
                elif lbl in LIAR_LEGITIMATE:
                    rows.append({'text': str(item['statement']).strip(), 'label': 0})
                # skip half-true
            return pd.DataFrame(rows)

        df_liar_train = liar_to_df(liar_raw['train'])
        print(f'\nLIAR train examples (after filtering): {len(df_liar_train)}')
        print(df_liar_train['label'].value_counts())
        df_liar_train.head()

except Exception as e:
    print(f'LIAR load failed: {e}')
    print('Continuing without LIAR dataset.')
    df_liar_train = pd.DataFrame(columns=['text', 'label'])

## 3d. Load additional datasets

- **liamvbetts/sarcastic-news-headlines-1** (26.7k) — sarcastic/outrage-bait headlines
- **daviddaubner/misinformation-detection** (26.1k) — misinformation and fear-mongering
- **FredZhang7/all-scam-spam** (42.6k) — scam/spam language patterns (filtered to short texts)

In [ ]:
df_extra_train = pd.DataFrame(columns=['text', 'label'])

extra_sources = []

# --- 1. Sarcastic news headlines (rage-bait proxy) ---
try:
    sarcasm_raw = load_dataset('liamvbetts/sarcastic-news-headlines-1')
    first_key = list(sarcasm_raw.keys())[0]
    rows = []
    for item in sarcasm_raw[first_key]:
        text = str(item.get('headline', '')).strip()
        if text:
            rows.append({'text': text, 'label': int(item.get('is_sarcastic', 0))})
    df_sarcasm = pd.DataFrame(rows)
    extra_sources.append(df_sarcasm)
    print(f'Sarcasm: {len(df_sarcasm)} examples')
    print(df_sarcasm['label'].value_counts())
except Exception as e:
    print(f'Sarcasm dataset failed: {e}')

# --- 2. Misinformation detection ---
try:
    misinfo_raw = load_dataset('daviddaubner/misinformation-detection')
    split_key = 'train' if 'train' in misinfo_raw else list(misinfo_raw.keys())[0]
    rows = []
    for item in misinfo_raw[split_key]:
        text = str(item.get('text', '')).strip()
        label = int(item.get('label', 0))
        if text:
            rows.append({'text': text, 'label': label})
    df_misinfo = pd.DataFrame(rows)
    extra_sources.append(df_misinfo)
    print(f'\nMisinformation: {len(df_misinfo)} examples')
    print(df_misinfo['label'].value_counts())
except Exception as e:
    print(f'Misinformation dataset failed: {e}')

# --- 3. Scam/spam (filtered to short texts only — closer to headline length) ---
try:
    scam_raw = load_dataset('FredZhang7/all-scam-spam')
    first_key = list(scam_raw.keys())[0]
    rows = []
    for item in scam_raw[first_key]:
        text = str(item.get('text', '')).strip()
        label = int(item.get('label', 0))
        # Keep only short texts (under 200 chars) to stay close to headline length
        if text and len(text) <= 200:
            rows.append({'text': text, 'label': label})
    df_scam = pd.DataFrame(rows)
    extra_sources.append(df_scam)
    print(f'\nScam/spam (short only): {len(df_scam)} examples')
    print(df_scam['label'].value_counts())
except Exception as e:
    print(f'Scam/spam dataset failed: {e}')

# Combine all extra sources
if extra_sources:
    df_extra_train = pd.concat(extra_sources, ignore_index=True)
    print(f'\nTotal extra examples: {len(df_extra_train)}')
    print(df_extra_train['label'].value_counts())
else:
    print('No extra datasets loaded.')

## 4. Add your own hand-labeled examples

These carry extra weight because they reflect the exact content the extension targets.
Add as many as you can — even 100 good examples meaningfully improve accuracy on edge cases.

`label: 1` = manipulative, `label: 0` = legitimate

In [7]:
# Load hand-labeled examples from the repo CSV.
# If running outside the cloned repo, upload data/custom_examples.csv
# using the Files panel on the left, then run this cell.

import os

CSV_PATH = 'data/custom_examples.csv'

if os.path.exists(CSV_PATH):
    df_custom = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(df_custom)} custom examples from {CSV_PATH}")
else:
    print(f"{CSV_PATH} not found.")
    print("Either clone the repo first, or upload data/custom_examples.csv via the Files panel.")
    print("Continuing with no custom examples — results will be weaker on edge cases.")
    df_custom = pd.DataFrame(columns=['text', 'label'])

print(df_custom['label'].value_counts())
df_custom.head()

data/custom_examples.csv not found.
Either clone the repo first, or upload data/custom_examples.csv via the Files panel.
Continuing with no custom examples — results will be weaker on edge cases.
Series([], Name: count, dtype: int64)


,text,label


In [ ]:
from sklearn.model_selection import train_test_split

# Repeat custom examples to give them more weight (3x)
df_custom_weighted = pd.concat([df_custom] * 3, ignore_index=True)

# If Webis wasn't available, carve a held-out test set from christinacdl
if not WEBIS_AVAILABLE and len(df_christinacdl_train) > 0:
    df_christinacdl_train, df_test = train_test_split(
        df_christinacdl_train, test_size=0.1, stratify=df_christinacdl_train['label'], random_state=42
    )
    print(f'No Webis test set — carved {len(df_test)} examples from christinacdl for testing.')

# Combine all sources
df_all_train = pd.concat(
    [df_train, df_christinacdl_train, df_liar_train, df_extra_train, df_custom_weighted],
    ignore_index=True
)
df_all_train = df_all_train.dropna(subset=['text'])
df_all_train = df_all_train[df_all_train['text'].str.strip().str.len() > 10]
df_all_train = df_all_train.drop_duplicates(subset=['text'])
df_all_train = df_all_train.sample(frac=1, random_state=42).reset_index(drop=True)

if len(df_all_train) == 0:
    raise RuntimeError('No training data loaded. Check that at least one dataset loaded successfully.')

# Validation split (if not provided by dataset)
if df_val is None:
    df_all_train, df_val = train_test_split(
        df_all_train, test_size=0.1, stratify=df_all_train['label'], random_state=42
    )

print(f'Train: {len(df_all_train)}  |  Val: {len(df_val)}  |  Test: {len(df_test)}')
print('\nLabel distribution (combined train):')
print(df_all_train['label'].value_counts())

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def df_to_hf_dataset(df):
    return Dataset.from_dict({
        'text':  df['text'].tolist(),
        'label': df['label'].tolist(),
    })

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

ds_train = df_to_hf_dataset(df_all_train).map(tokenize, batched=True)
ds_val   = df_to_hf_dataset(df_val).map(tokenize, batched=True)
ds_test  = df_to_hf_dataset(df_test).map(tokenize, batched=True)

ds_train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
ds_val.set_format('torch',   columns=['input_ids', 'attention_mask', 'label'])
ds_test.set_format('torch',  columns=['input_ids', 'attention_mask', 'label'])

print("Tokenization done.")
print("Sample token count:", len(ds_train[0]['input_ids']))

## 7. Define model and metrics

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'legitimate', 1: 'manipulative'},
    label2id={'legitimate': 0, 'manipulative': 1},
)
model.to(device)
print(f"Model parameters: {model.num_parameters():,}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='binary'),
    }

## 8. Train

On a free Colab T4 GPU this takes about 20–30 minutes.

In [ ]:
collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir                  = './checkpoints',
    num_train_epochs            = 4,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    learning_rate               = 2e-5,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    logging_steps               = 100,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = ds_train,
    eval_dataset    = ds_val,
    tokenizer       = tokenizer,
    data_collator   = collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 9. Evaluate on test set

In [ ]:
results = trainer.predict(ds_test)
preds   = np.argmax(results.predictions, axis=-1)
labels  = results.label_ids

print(classification_report(labels, preds, target_names=['legitimate', 'manipulative']))

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['legitimate', 'manipulative'],
            yticklabels=['legitimate', 'manipulative'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Test Set Confusion Matrix')
plt.tight_layout()
plt.show()

## 10. Spot-check predictions

Run your own headlines through the model before exporting.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

test_headlines = [
    "Is it a problem if the Fed speaks too much?",
    "You won't believe what happened next",
    "Severe storms, tornadoes slam over 1,000 miles across Plains, Midwest",
    "Over $70B in Unclaimed Funds in U.S. Enter Name & State.",
    "Senator SLAMS colleague in explosive hearing",
    "NASA confirms Perseverance rover collected 10th Mars rock sample",
    # Add your own here:
]

for h in test_headlines:
    result = classifier(h)[0]
    label  = result['label']
    score  = result['score']
    flag   = '🚩' if label == 'manipulative' else '  '
    print(f"{flag} [{score:.2f}] {h}")

## 11. Save model

Save the fine-tuned model and tokenizer locally (in Colab), then download the folder.

In [ ]:
SAVE_DIR = './headline-classifier-model'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}")

## 12. Export to ONNX (for Chrome extension via Transformers.js)

Transformers.js can run ONNX models entirely in the browser — no server needed.  
This exports + quantizes the model to ~25MB (int8).

In [ ]:
!optimum-cli export onnx \
    --model ./headline-classifier-model \
    --task text-classification \
    --opset 14 \
    ./headline-classifier-onnx

In [ ]:
# Quantize to int8 — shrinks from ~250MB to ~65MB
from optimum.onnxruntime import ORTModelForSequenceClassification
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from optimum.onnxruntime import ORTQuantizer

quantizer = ORTQuantizer.from_pretrained('./headline-classifier-onnx')
qconfig   = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)

quantizer.quantize(
    save_dir           = './headline-classifier-onnx-quantized',
    quantization_config= qconfig,
)
print("Quantization done.")

import os
for f in os.listdir('./headline-classifier-onnx-quantized'):
    size = os.path.getsize(f'./headline-classifier-onnx-quantized/{f}') / 1e6
    print(f"  {f}: {size:.1f} MB")

In [ ]:
# Zip the quantized model for download
!zip -r headline-classifier-onnx-quantized.zip ./headline-classifier-onnx-quantized
print("Download headline-classifier-onnx-quantized.zip from the Files panel on the left.")

## 13. Quick ONNX inference test

Confirm the exported model produces the same results before downloading.

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import pipeline, AutoTokenizer

ort_model    = ORTModelForSequenceClassification.from_pretrained('./headline-classifier-onnx-quantized')
ort_tok      = AutoTokenizer.from_pretrained('./headline-classifier-onnx-quantized')
ort_pipe     = pipeline('text-classification', model=ort_model, tokenizer=ort_tok)

for h in test_headlines:
    result = ort_pipe(h)[0]
    flag   = '🚩' if result['label'] == 'manipulative' else '  '
    print(f"{flag} [{result['score']:.2f}] {h}")

## Next steps

1. **Download** `headline-classifier-onnx-quantized.zip` from the Files panel
2. **Bundle** the ONNX model into the Chrome extension alongside `@xenova/transformers` (Transformers.js)
3. **Call** the model from the content script instead of the regex pattern library
4. **Iterate** — add more hand-labeled examples and retrain to improve edge cases

Transformers.js docs: https://huggingface.co/docs/transformers.js